In [1]:
from transformers import AutoProcessor
import json

In [32]:
prompts = json.load(open("../../../curated_data/text/text_dataset/text_concept_a_female_person_v2.json"))
vis = json.load(open("../../../activations/text/female_vis_direct_prompt.json"))
processor = AutoProcessor.from_pretrained("google/gemma-3-27b-it")

FileNotFoundError: [Errno 2] No such file or directory: '../../../curated_data/text/text_dataset/text_concept_a_female_person_v2.json'

In [33]:
from circuitsvis.tokens import colored_tokens_multi
import torch



def visualize(features, prompts,n_batches, batch_size, layer,chosen_concept, start_from=0, system_prompt_token_count=0):
    pad = [0.0]*len(features[str(layer)][0][0][0])
    batched = features[str(layer)][start_from].copy()
    for i in range(start_from+1,start_from+n_batches):
        batched += features[str(layer)][i].copy()

    max_len = len(max(batched, key=lambda x: len(x)))
    for item in batched:
        if len(item)<max_len:
            for _ in range(max_len-len(item)):
                item.insert(0,pad.copy())        

    messages = [[
                    {
                        "role": "system",
                        "content": [{"type": "text", "text": f"You are a linguistics expert, your task is to identify all words that fall under the linguistic umbrella of {chosen_concept}, whether that manifests in direct words, nouns, pronouns, etc"}]
                    },
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": prompts[i]}
                        ]
                    }
                ] for i in range(start_from*batch_size, n_batches*batch_size+start_from*batch_size)]
    # print(list(range(start_from*batch_size, n_batches*batch_size+start_from*batch_size)), list(range(start_from+1,start_from+n_batches)))
    tokens = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True,
                    return_dict=True, return_tensors="pt", padding=True)
    str_tokens = [processor.decode(t) for t in tokens["input_ids"][:,system_prompt_token_count:].flatten(end_dim=1)]
    # Visualize activations for top 20 most prominent features
    return colored_tokens_multi(str_tokens, torch.tensor(batched, dtype=torch.float32)[:,system_prompt_token_count:,:].flatten(end_dim=1))



In [34]:
rendered = visualize(features=vis.copy(), prompts=prompts.copy(), n_batches=8, batch_size=2, layer=59,chosen_concept="A female person", start_from=0)

In [35]:
rendered